# Propiedades

Notebook para la implementación indiviual de propiedades para su posterior integración con el resto del código.

In [2]:
import networkx as nx

### "boilerplate code", no modificar ⚠️

In [3]:
from abc import ABC, abstractmethod

# Decorators for graph characteristics

def use_direction(cls):
    cls._use_direction = True
    return cls

def use_selfloops(cls):
    cls._use_selfloops = True
    return cls

def use_giant_component(cls):
    cls._use_giant_component = True
    return cls

def return_scalar(cls):
    cls._return_type = "scalar"
    return cls

def return_distribution(cls):
    cls._return_type = "distribution"
    return cls

def use_paths(cls):
    cls._use_paths = True
    return cls


class _Property(ABC):
    """Abstract base class for all properties."""
    _return_type = None
    _use_paths = False
    _use_direction = False
    _use_selfloops = False
    _use_giant_component = False

    def __init__(self, G: nx.DiGraph):
        self.G = G
        self._raw_value = None

    @abstractmethod
    def compute(self):
        return self._raw_value

    @abstractmethod
    def norm_biol(self, *args):
        pass

    @abstractmethod
    def norm_network(self, *args):
        pass




In [45]:
class NormalizationError(Exception):
    """Exception raised for errors in the normalization.

    Attributes:
        message -- explanation of the error
    """

    def __init__(self, message="An error occurred during normalization."):
        self.message = message
        super().__init__(self.message)

class NullGraphError(Exception):
    """Exception raised for null graph."""
    pass


def check_raw_value(func):
    """Decorator to check if raw value is None. If it is, raise an error."""
    def wrapper(self, *args):
        if self._raw_value is not None:
            return func(self, *args)
        else:
            raise ValueError("Raw value is None. Call compute() method first.")
    return wrapper

### Describimos la propiedad particular:
Incluye al menos una referencia de su uso previo, de preferencia en redes biológicas.
Para incluir citas, usa el PubMed ID (PMID) de la publicación entre corchetes. Para múltiples citas, asigna sólo una detro de cada par de corchetes, sin espacios entre citas: [PMID1][PMID2][PMID3]

    TU DEFINICIÓN AQUÍ 👇

The network density is the ratio of the number of edges in the network to the total number of possible edges in the network.
For a directed network with self loops, the total number of possible edges is $n^2$, where $n$ is the number of nodes in the network. The density takes values from 0 for a network with no edges, to 1 for a fully connected networ (complete graph).

In bacterial gene regulatory networks, the density is constrained to very small values (~1e-2) [30842463]. This is, only around one percent of the potential interactions (all genes regulating each other) ocur in the regulation machinery. For gene regulatory networks, we know the total number of possible edges is not $n*n$ beacuse only a fraction of nodes code for transcription factors. Instead, considering this biological constrain, the maximum total of potential edges is given by $tf*n$, where $tf$ is the number of genes coding for transcription factors in the newtork and $tf$&leq;$n$. Therefore, with this biological consideration, the formula $m/(tf*n)$ yields greater densities.

### Definimos la clase hija y sus métodos o funciones auxiliares:

    YOUR CODE HERE 👇

In [61]:
# Esta función auxiliar no se incluyó como método de la clase Density porque se utilizará
# cada vez que se requiera la lista de TFs
def get_parent_nodes(G: nx.DiGraph):
    """Get the parent nodes of a graph."""
    return [i for i, k_out in G.out_degree() if k_out > 0]


# Los decoradores que definan la característica de la red con la que se calcula la propiedad.
# Sólo incluir los decoradores que se deban evaluar a True, el resto mantienen su valor por defecto (False).
# Las opciones son:
#   - @use_direction
#   - @use_selfloops
#   - @use_giant_component
#   - @use_paths
#   - @return_scalar
#   - @return_distribution
# Either @return_scalar or @return_distribution must be used, not both.
@return_scalar 
@use_direction
@use_selfloops
@use_giant_component    
class Density(_Property): # Hereda de la clase _Property
    """Density of the graph.

    The density of a graph is defined as the ratio of the number of edges to the number of possible edges.
    The number of possible edges for directed network with self loops is given by n**2,
    where n is the number of nodes in the graph.

    Methods:
        compute: Compute the density of the graph.
        norm_biol: Normalize the density of the graph to the number of parents.
        norm_network: Normalize the density of the graph to the number of nodes.
    """

    def __init__(self, G: nx.DiGraph):
        """
        Args:
            G (nx.DiGraph): Graph.
        """
        super().__init__(G)
        self._n_nodes = self.G.number_of_nodes()
        if self._n_nodes == 0:
            raise NullGraphError("A null graph has no density.")

    def compute(self) -> float:
        """Compute the density of the graph.

        Returns:
            float: Density of the graph.    
        """
        n_edges = self.G.number_of_edges()
        self._raw_value = n_edges / self._n_nodes**2
        return self._raw_value

    @check_raw_value    # Decorator to check if raw value is None. If it is, raise an error.
    def norm_biol(self) -> float:
        """Normalize the density of the graph to the number of parents."""
        n_parents = len(get_parent_nodes(self.G))
        try:
            return self._raw_value * (self._n_nodes / n_parents)
        except ZeroDivisionError:
            raise NormalizationError("Division by zero (no parent nodes). Cannot normalize with this approach.")

    @check_raw_value
    def norm_network(self) -> float:
        """Normalize the density of the graph to the number of nodes. (Already normalized)"""
        return self._raw_value  # density is already normalized to [0,1]

In [63]:
@return_distribution
@use_direction
@use_selfloops
class Degree():
    pass

In [64]:
Degree

### Pruebas de tu código
Agrega evaluaciones a tu código con redes que se conoce su valor teórico. Un grafo vacío y uno completo es de los casos más comunes, pero puedes agregar los que consideres necesario.

    TUS PRUEBAS AQUÍ 👇

In [59]:
# Para densidad usaremos un grafo completo dirigido con self loops y uno sin ninguna interacción, sólo nodos.
import pytest
import networkx as nx

# null graph
G = nx.DiGraph()
with pytest.raises(NullGraphError) as e_info:
    property = Density(G)

# empty graph with nodes
G = nx.DiGraph()
n_nodes = 10
G.add_nodes_from(range(n_nodes))    # add nodes, no edges

# no edges
property = Density(G)
assert property.compute() == 0 # no edges (0 / 10*10)
with pytest.raises(NormalizationError) as e_info:
    property.norm_biol() # no edges (0 / 10*10) * (10 / 0)
assert property.norm_network() == 0 # no edges (0 / 10*10)

# add edges
# complete directed graph with self loops
G.add_edges_from([(i, j) for i in range(n_nodes) for j in range(n_nodes)])
property = Density(G)
assert property.compute() == 1 # all possible edges (100 / 10*10)
assert property.norm_biol() == 1 # all possible edges (100 / 10*10) * (10 / 10)
assert property.norm_network() == 1 # all possible edges (100 / 10*10)


# only half of the nodes are parents and regulate every node in the graph
G = nx.DiGraph()
n_nodes = 10
G.add_nodes_from(range(n_nodes))
G.add_edges_from([(i, j) for i in range(n_nodes//2) for j in range(n_nodes)])
property = Density(G)
assert property.compute() == 0.5 # half of the possible edges (50 / 10*10)
assert property.norm_biol() == 1 # all of the possible edges considering the other half are structural genes (50 / 10*10) * (10 / 5)
assert property.norm_network() == 0.5 # half of the possible edges (50 / 10*10)
